# Verifica con Nagini del numero di porte: algoritmo di Grover

 ## Notazione
 - n: numero di qubit del circuito (qubit di stato)
 - m: numero di qubit su cui agisce la riflessione, cioè `len(qubits)`
 - |O|: numero di porte dell'oracolo
 - k: numero di iterazioni di Grover  

## Premessa


- Cominciamo ad analizzare il codice di `GroverOperator`, che costruisce una **singola iterazione di Grover**
- Una singola iterazione applica in ordine:
    - l'oracolo, che marca le soluzioni con un segno -1
    - la trasformata di Hadamard $H^{⊗n}$
    - uno spostamento di fase condizionale, che lascia invariato $|0…0⟩$ e cambia segno a tutti gli altri stati
    - un'ulteriore trasformata di Hadamard $H^{⊗n}$

- Qiskit scrive l'iterazione come $Q = A S_0 A† S_f$, dove:
    -  $S_f$ è l'oracolo, 
    - $A = H^{⊗n}$ è la preparazione di stato 
    - $S_0$ è lo spostamento di fase condizionale (riflessione attorno a $|0…0⟩$)
- Gli ultimi tre passi insieme, $A S_0 A†$, formano l'inversione attorno alla media $2|ψ⟩⟨ψ| - I$


### Struttura di `GroverOperator`

(file `grover_originale/grover_operator_originale.py`)

- `__init__`: 
    - salva i parametri (oracolo, preparazione di stato, riflessione, qubit di riflessione, `insert_barriers`, `mcx_mode`) e chiama `_build`. 
    - Se l'oracolo o la riflessione sono dati come `Statevector` o `Operator`, li converte in un circuito `Diagonal`

- `reflection_qubits`: 
    - i qubit su cui agisce la riflessione $S_0$ di default tutti i qubit di stato, cioè `[0, ..., n-1]`
- `zero_reflection`: 
    - restituisce $S_0$. 
    - Se l'utente non l'ha fornita, la costruisce chiamando la funzione `_zero_reflection`
- `state_preparation`: 
    - restituisce A. 
    - Se l'utente non l'ha fornita, costruisce un circuito con H sui qubit di riflessione
- `oracle`: 
    - restituisce l'oracolo passato dall'utente, che marca con -1 le soluzioni. 
- `_build`: 
    - compone l'iterazione completa nell'ordine oracolo, $A†$ (`state_preparation.inverse()`), $S_0, A$
    - imposta la fase globale a $π$ per compensare il segno di $S_0$ e infine impacchetta tutto in un unico gate con `to_gate()`

**Ordine della verifica**
- Si parte dalle funzioni che non dipendono da altre: 
    1. `_zero_reflection` 
    2. la preparazione di stato
    3. Poi `_build`, che le usa tramite le loro postcondizioni
   4. Infine `construct_circuit` della classe `Grover`, che ripete k volte l'iterazione

# 1. `_zero_reflection`

- Parto da questa funzione perchè non è richiamata da altre.
- Costruisce $S_0$, la riflessione attorno a $|0…0⟩$ sui qubit indicati. 
    - l'operazione da svolgere è $2|0⟩⟨0| - I$ 
    - viene impelentato $I - 2|0⟩⟨0|$ perchè è più semplice e perchè differisce di una fase globale (stato quantistico è definito a meno di una fase globale) 


## 1.1 Codice originale

File `grover_originale/grover_operator_originale.py`, righe 289-311 (Qiskit, licenza Apache 2.0).

In [ ]:
def _zero_reflection(
    num_state_qubits: int, qubits: List[int], mcx_mode: Optional[str] = None
) -> QuantumCircuit:
    qr_state = QuantumRegister(num_state_qubits, "state")
    reflection = QuantumCircuit(qr_state, name="S_0")

    num_ancillas = MCXGate.get_num_ancilla_qubits(len(qubits) - 1, mcx_mode)
    if num_ancillas > 0:
        qr_ancilla = AncillaRegister(num_ancillas, "ancilla")
        reflection.add_register(qr_ancilla)
    else:
        qr_ancilla = AncillaRegister(0)

    reflection.x(qubits)
    if len(qubits) == 1:
        reflection.z(0)  # MCX does not allow 0 control qubits, therefore this is separate
    else:
        reflection.h(qubits[-1])
        reflection.mcx(qubits[:-1], qubits[-1], qr_ancilla[:], mode=mcx_mode)
        reflection.h(qubits[-1])
    reflection.x(qubits)

    return reflection

## 1.2 Limite da dimostrare  

3 porte se m = 1 e 2m + 3 porte se m ≥ 2

- il numero di porte dipende solo da m, non dal numero totale di qbit del circuito

- In entrambi i casi il circuito è racchiuso tra due strati di X (bit flip) su tutti i qubit di riflessione: 2m porte
    - servono a scambiare |0…0⟩ e |1…1⟩, così il cambio di segno finisce su |0…0⟩
 
- In mezzo serve una porta che cambi segno solo a |1…1⟩ (phase flip)
- **Caso m = 1**: basta una Z sul qubit
    - totale: 2 X + 1 Z = 2m + 1 = 3 porte
- **Caso m ≥ 2**: serve una Z multi-controllata (controlli: tutti i qubit tranne l'ultimo; target: l'ultimo)
    - Qiskit la costruisce come H, MCX, H sul target, grazie all'uguaglianza HXH = Z
    - totale: 2m X + H + MCX + H = 2m + 3 porte

## 1.3 Metodi aggiunti allo stub
- il punto precedente fa vedere che vanno aggiunti dei metodi allo stub che era stato fatto per Deutsch. 
- conteneva x, h, cx 

**METODO Z** identico a h

In [ ]:
def z(self, qubit: int) -> None:
    Requires(Acc(self.num_qubits, 1/2) and Acc(self.gates))
    Requires(0 <= qubit and qubit < self.num_qubits)

    Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))
    Ensures(self.gates == Old(self.gates) + 1)

    self.gates = self.gates + 1

**METODO X_LIST**
- Corrisponde a `x(lista)` di Qiskit, che aggiunge una porta X per ogni qubit della lista
- È un metodo separato da `x` per evitare problemi di tipo con Nagini (un parametro che può essere sia intero sia lista)
- Chiede 1/4 del permesso sulla lista, non 1/2

**Causa**
- `_zero_reflection` ha 1/2 del permesso su `qubits`; nella prima versione `x_list` chiedeva proprio 1/2, quindi la funzione cedeva tutto il suo permesso
- Chi cede tutto il permesso, al ritorno sa della lista solo ciò che dice la postcondizione del metodo chiamato
- La postcondizione di `x_list` non dice che la lista è rimasta uguale: Nagini ne dimentica contenuto e lunghezza, e alla seconda chiamata non riesce più a dimostrare che gli indici sono validi

**Correzione**
- `x_list` (e `mcx`) chiedono 1/4; `_zero_reflection` resta a 1/2
- Così la funzione cede 1/4 e tiene l'altro 1/4: la lista non può essere modificata e Nagini continua a conoscerne il contenuto

In [ ]:
def x_list(self, qubits: List[int]) -> None:
        Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Requires(Acc(list_pred(qubits), 1/4))
        Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < self.num_qubits)))

        Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Ensures(Acc(list_pred(qubits), 1/4))
        Ensures(self.gates == Old(self.gates) + len(qubits))

        self.gates = self.gates + len(qubits)

**METODO MCX:**
- Una sola porta, qualunque sia il numero di controlli
- `len(controls) >= 1`: MCX richiede almeno un controllo; è il motivo per cui l'originale tratta a parte il caso m = 1
- Non richiede che il target sia diverso dai controlli (semplificazione, vedi 1.8)

In [ ]:
def mcx(self, controls: List[int], target: int) -> None:
        Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Requires(Acc(list_pred(controls), 1/4))
        Requires(len(controls) >= 1)
        Requires(Forall(int, lambda i: Implies(0 <= i and i < len(controls), 0 <= controls[i] and controls[i] < self.num_qubits)))
        Requires(0 <= target and target < self.num_qubits)

        Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Ensures(Acc(list_pred(controls), 1/4))
        Ensures(self.gates == Old(self.gates) + 1)

        self.gates = self.gates + 1

## 1.4 Trasformazioni

- **Parametro `mcx_mode`**
    - Originale: parametro della funzione
    - Versione verificata: rimosso, fissato a `"noancilla"`
    - Motivo: è il default di `GroverOperator`
- **Gestione delle ancille**
    - Originale: `MCXGate.get_num_ancilla_qubits(...)` e blocco `if` con `AncillaRegister`
    - Versione verificata: rimossi
    - Motivo: con `"noancilla"` le ancille sono sempre 0, quindi si prende sempre il ramo senza ancille
- **Costruzione del circuito**
    - Originale: `QuantumCircuit(QuantumRegister(n, "state"), name="S_0")`
    - Versione verificata: `QuantumCircuit(n, 0)`
    - Motivo: il modello non ha registri né nomi; il secondo argomento sono i bit classici
- **X su una lista di qubit**
    - Originale: `reflection.x(qubits)`
    - Versione verificata: `reflection.x_list(qubits)`
    - Motivo: un metodo per ogni tipo di argomento
- **Z nel caso m = 1**
    - Originale: `reflection.z(0)`
    - Versione verificata: `reflection.z(qubits[0])`
    - Motivo: correzione di un errore del codice originale (vedi 1.8)
- **Target di MCX**
    - Originale: `qubits[-1]`
    - Versione verificata: `qubits[len(qubits) - 1]`, salvato in `target`
    - Motivo: indici espliciti, non negativi
- **Controlli di MCX**
    - Originale: `qubits[:-1]`
    - Versione verificata: `qubits[:len(qubits) - 1]`, salvato in `controls`
    - Motivo: come sopra
- **Chiamata a MCX**
    - Originale: `mcx(..., qr_ancilla[:], mode=mcx_mode)`
    - Versione verificata: `mcx(controls, target)`
    - Motivo: il modello non ha ancille né modalità

**Note**
- Le riscritture di indici negativi e slicing sono preventive: la forma originale non è stata provata con Nagini
- Le variabili `target` e `controls` servono solo alla leggibilità

mcx considerata come un unica istruzione
- Sia il modello sia `size()` di Qiskit contano MCX come un'unica istruzione
- Senza ancille la scomposizione richiede molte più porte; con le ancille ne richiede meno, al prezzo di qubit in più
- Se si decidesse di contare la scomposizione, il limite dipenderebbe sia da m sia dalla modalità: è una delle decisioni aperte

## 1.5 Versione verificata

File `groverVerificato.py`.

In [ ]:
def _zero_reflection(num_state_qubits: int, qubits: List[int]) -> QuantumCircuit:
    Requires(num_state_qubits > 0)
    Requires(Acc(list_pred(qubits), 1/2))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < num_state_qubits)))

    Ensures(Acc(list_pred(qubits), 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == num_state_qubits)
    Ensures(Implies(len(qubits) == 1, Result().gates == 3))
    Ensures(Implies(len(qubits) > 1, Result().gates == 2 * len(qubits) + 3))
    Ensures(Result().num_clbits == 0)

    reflection = QuantumCircuit(num_state_qubits, 0)

    reflection.x_list(qubits)
    if len(qubits) == 1:
        reflection.z(qubits[0])
    else:
        target = qubits[len(qubits) - 1]
        controls = qubits[:len(qubits) - 1]
        reflection.h(target)
        reflection.mcx(controls, target)
        reflection.h(target)
    reflection.x_list(qubits)

    return reflection

# 2. `_state_preparation`
- Costruisce A, la preparazione di stato: una porta H su ciascun qubit di riflessione
    - corrisponde alle due trasformate di Hadamard dell'iterazione di Grover, prima e dopo S_0
    - in `_build` viene usata due volte: invertita (A†) prima di S_0 e normale (A) dopo
    - siccome H† = H, qui A† e A contengono le stesse porte
- Come `_zero_reflection`, non chiama altre funzioni di Grover, quindi si può verificare direttamente


## 2.1 Codice originale
File `grover_originale/grover_operator_originale.py`, righe 227-237 (Qiskit, licenza Apache 2.0). Non è una funzione libera ma una property della classe `GroverOperator`.

In [ ]:
@property
def state_preparation(self) -> QuantumCircuit:
    """The subcircuit implementing the A operator or Hadamards."""
    if self._state_preparation is not None:
        return self._state_preparation

    num_state_qubits = self.oracle.num_qubits - self.oracle.num_ancillas
    hadamards = QuantumCircuit(num_state_qubits, name="H")
    # apply Hadamards only on reflection qubits, rest will cancel out
    hadamards.h(self.reflection_qubits)
    return hadamards

**Cosa fa**

- Se l'utente ha fornito una sua preparazione di stato, la restituisce così com'è
- Altrimenti calcola il numero di qubit di stato dall'oracolo e crea un circuito con H sui qubit di riflessione
- Il commento spiega perché le H vanno solo sui qubit di riflessione: sugli altri qubit S_0 non agisce, quindi le H di A† e di A si annullerebbero (H H = I)

## 2.2 Limite da dimostrare

m porte, con m = len(qubits)

- Una H per ogni qubit di riflessione
- Il numero di porte dipende solo da m, non dal numero totale di qubit del circuito
- Non ci sono casi separati: la formula vale per ogni m ≥ 1

## 2.3 Metodi aggiunti allo stub

**METODO H_LIST**

- Corrisponde a `h(lista)` di Qiskit: aggiunge una porta H per ogni qubit della lista
- È identico a `x_list`, cambia solo il nome
- Chiede 1/4 del permesso sulla lista per lo stesso motivo di `x_list` (vedi 1.3)

In [ ]:
def h_list(self, qubits: List[int]) -> None:
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Requires(Acc(list_pred(qubits), 1/4))
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < self.num_qubits)))

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Ensures(Acc(list_pred(qubits), 1/4))
    Ensures(self.gates == Old(self.gates) + len(qubits))

    self.gates = self.gates + len(qubits)

## 2.4 Trasformazioni

- **Da property a funzione libera**
    - Originale: `@property def state_preparation(self)`
    - Versione verificata: `_state_preparation(num_state_qubits, qubits)`
    - Motivo: la property legge tutti i dati da `self` (oracolo, qubit di riflessione, preparazione dell'utente), ma il numero di porte dipende solo dal numero di qubit e dalla lista dei qubit. Passandoli come parametri si evita di modellare l'intera classe `GroverOperator` e i contratti restano piccoli
- **Preparazione fornita dall'utente**
    - Originale: `if self._state_preparation is not None: return self._state_preparation`
    - Versione verificata: rimosso
    - Motivo: una preparazione arbitraria ha un numero di porte non noto; si considera il Grover standard con le H
- **Numero di qubit di stato**
    - Originale: `self.oracle.num_qubits - self.oracle.num_ancillas`
    - Versione verificata: parametro `num_state_qubits`
    - Motivo: senza ancille coincide con il numero di qubit dell'oracolo; sarà `_build` a passarlo
- **Qubit di riflessione**
    - Originale: `self.reflection_qubits`
    - Versione verificata: parametro `qubits`
    - Motivo: come sopra; `_build` passa la stessa lista a `_state_preparation` e a `_zero_reflection`
- **Costruzione del circuito**
    - Originale: `QuantumCircuit(num_state_qubits, name="H")`
    - Versione verificata: `QuantumCircuit(num_state_qubits, 0)`
    - Motivo: il modello non ha nomi; il secondo argomento sono i bit classici
- **H su una lista di qubit**
    - Originale: `hadamards.h(self.reflection_qubits)`
    - Versione verificata: `hadamards.h_list(qubits)`
    - Motivo: un metodo per ogni tipo di argomento

**Note**

- La trasformazione è corretta perché la funzione esegue le stesse operazioni della property, a patto di riceverle con gli stessi valori che la property leggeva da `self`: `num_state_qubits` uguale al numero di qubit dell'oracolo e `qubits` uguale ai qubit di riflessione
- Regola generale, utile per il traduttore automatico: una property o un metodo che usa `self` solo in lettura diventa una funzione libera che riceve quei dati come parametri
- `_zero_reflection` non ha richiesto questo passaggio perché in Qiskit era già una funzione libera

## 2.5 Versione verificata
File `groverVerificato.py`.

In [ ]:
def _state_preparation(num_state_qubits: int, qubits: List[int]) -> QuantumCircuit:
    Requires(num_state_qubits > 0)
    Requires(Acc(list_pred(qubits), 1/2))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits),
                                           0 <= qubits[i] and qubits[i] < num_state_qubits)))

    Ensures(Acc(list_pred(qubits), 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == num_state_qubits)
    Ensures(Result().gates == len(qubits))
    Ensures(Result().num_clbits == 0)

    hadamards = QuantumCircuit(num_state_qubits, 0)
    hadamards.h_list(qubits)
    return hadamards

**Contratti**

- Le precondizioni sono le stesse di `_zero_reflection`, perché le due funzioni ricevono gli stessi dati
- Il costruttore parte da 0 porte e `h_list` ne aggiunge len(qubits): da qui `Result().gates == len(qubits)`
- `Result().num_clbits == 0` è stata aggiunta durante il livello 3: in `_build` il risultato di `_state_preparation` viene invertito con `inverse()`, che richiede `num_clbits >= 0`
- La verifica è passata al primo tentativo: le regole sui permessi ricavate al livello 1 sono bastate